#### Workflow Objectives: 
The primary focus of this phase is to filter artifacts and validate the biological integrity of the ATAC-seq library. TThe workflow integrates multiple complementary tools across four core stages:

* **Deduplication & Formatting**: Identify and mark PCR/optical duplicates using Picard 2.26.10 MarkDuplicates to reduce over-amplification bias.
* **Rigorous Filtering**: Generate filtered BAM files using Samtools 1.17 to retain only high-quality, properly paired reads..
* **ATAC-specific QC**: Evaluate fragment size distribution and alignment quality metrics using R package, ATACseqQC 1.28.0.
* **Signal Normalization and Visualization**: Generate normalized genome-wide accessibility tracks and enrichment profiles using deepTools.

#### Data Processing & Implementation
All tasks are submitted to the HPC cluster using the **SLURM scheduler** (default: 4 GB memory, 1 CPU, 6-hour runtime). Each step is executed as a separate job script to ensure efficient resource allocation and reproducibility. Following the generation of the merged and filtered BAM files, downstream quality control and statistical assessments are performed using the ATACseqQC (v1.28.0) package within the R/Bioconductor environment. 

#### Step 1: Deduplication & Formatting
Automates the conversion from SAM to BAM and marks duplicate reads.

1a. Sort and index bam files using samtools

In [ ]:
#!/bin/bash
# Standardize alignment files and generate mapping statistics.

#SBATCH --cpus-per-task=8
#SBATCH --mem=8G
#SBATCH --output=logs/sort_index_%A_%a.out
#SBATCH --error=logs/sort_index_%A_%a.err

# Load Bowtie2 module (required if using a module-based system)
module load samtools/1.17

# Create the output directory
mkdir -p bowtie2_results/sorted_bam

# Define an array of file names to be processed
readarray -t files < <(ls fastq_files/*.fastq.gz | sed 's/.*fastq_files//' | sed 's/_R.*//' | sort -u)

# Get the file name for this array task
file="${files[$SLURM_ARRAY_TASK_ID - 1]}"

# Perform sorting and indexing for the specific file
srun ~/tools/samtools-1.17/samtools view -bS bowtie2_results${file}.sam | ~/tools/samtools-1.17/samtools sort -o bowtie2_results/sorted_bam${file}.sorted.bam && srun ~/tools/samtools-1.17/samtools index bowtie2_results/sorted_bam${file}.sorted.bam

# Generate statistics using samtools stats
srun ~/tools/samtools-1.17/samtools stats bowtie2_results/sorted_bam${file}.sorted.bam > bowtie2_results/sorted_bam${file}_stats.txt

In [ ]:
# Run MultiQC
conda activate python3.7
multiqc ./bowtie2_results/sorted_bam/. -o ./bowtie2_results/sorted_bam/
conda deactivate 

<img src="https://www.dropbox.com/scl/fi/qstusy6jcy2wnw21n2yd4/samtools_alignment_plot-6.png?rlkey=1z4evrru9vu49mhy3njq52hf5&st=uknjshct&raw=1" width="600" alt="samtool_sorting">

* The majority of samples achieved high mapping rates (typically $>50\text{M}$ reads), ensuring sufficient coverage for high-resolution chromatin accessibility analysis.
* While a small number of samples exhibited a higher proportion of unmapped reads, the unique concordant alignments across the cohort remained robust


1b. Mark duplicates using Picard

In [ ]:
#!/bin/bash
# Mark PCR and optical duplicates to address over-amplification bias.

#SBATCH --cpus-per-task=8
#SBATCH --mem=16G
#SBATCH --time=08:00:00
#SBATCH --output=logs/markdup_%A_%a.out
#SBATCH --error=logs/markdup_%A_%a.err

# Load Picard module (required if using a module-based system)
module load picard/2.26.10-Java-15.lua

# Create the output directory
mkdir -p bowtie2_results/markdup_bam

# Define an array of file names to be processed
readarray -t files < <(ls fastq_files/*.fastq.gz | sed 's/.*fastq_files//' | sed 's/_R.*//' | sort -u)

# Get the file name for this array task
file="${files[$SLURM_ARRAY_TASK_ID - 1]}"

# Perform bowtie2 alignment for the specific file
srun java -jar $EBROOTPICARD/picard.jar MarkDuplicates -I bowtie2_results/sorted_bam${file}.sorted.bam -O bowtie2_results/markdup_bam${file}_marked_duplicates.bam -M bowtie2_results/markdup_bam${file}_marked_dup_metrics.txt

In [ ]:
# Run MultiQC
conda activate python3.7
multiqc ./bowtie2_results/markdup_bam/. -o ./bowtie2_results/markdup_bam/
conda deactivate 

<img src="https://www.dropbox.com/scl/fi/xkeaihdnf79utew334y0k/picard_deduplication-1.png?rlkey=uovnke4ngymp6qdf11drly16n&st=mwtiglme&raw=1" width="600" alt="picard_markdup">

* Most samples maintained high library complexity with a significant proportion of unique pairs. 
* While non-optical duplicates (orange) were present across the cohort—typical for ATAC-seq libraries—the high ratio of unique reads confirms sufficient library diversity for high-resolution peak calling.

#### Step 2: Rigorous Filtering
Applies specific ATAC-seq filters to retain only high-quality nuclear reads.

2a. Filter bam files (mitochondrial reads, secondary alignments, non-unique alignment, unmapped, mate unmapped, not primary alignment, reads failing platform, duplicates and keeps only the properly paired reads)

In [ ]:
#!/bin/bash
# Exclude mitochondrial reads (chrM). Remove secondary alignments and low-quality reads (MAPQ < 10). Retain only properly paired, primary alignments.

#SBATCH --cpus-per-task=8
#SBATCH --mem=8G
#SBATCH --output=logs/filter_%A_%a.out
#SBATCH --error=logs/filter_%A_%a.err

# Create the output directory
mkdir -p bowtie2_results/filtered_bam

# Define an array of file names to be processed
readarray -t files < <(ls fastq_files/*.fastq.gz | sed 's/.*fastq_files//' | sed 's/_R.*//' | sort -u)

# Get the file name for this array task
file="${files[$SLURM_ARRAY_TASK_ID - 1]}"

# Filter out mitochondrial reads
# Filters out reads that are marked as secondary alignments (1024)
# Filters out reads with mapping quality less than 10 (q < 10) or non-unique alignment
# Filters out reads that are marked as unmapped, mate unmapped, not primary alignment, reads failing platform, duplicates and keeps only the properly paired reads (1804, -f 2)
srun ~/tools/samtools-1.17/samtools view -h bowtie2_results/markdup_bam${file}_marked_duplicates.bam \
    | grep -v "chrM" \
    | ~/tools/samtools-1.17/samtools view -F 1024 -b \
    | ~/tools/samtools-1.17/samtools view -q 10 -b \
    | ~/tools/samtools-1.17/samtools view -F 1804 -f 2 -b \
    | ~/tools/samtools-1.17/samtools sort -o bowtie2_results/filtered_bam${file}_quality_controlled.bam

# Generate statistics using samtools stats
srun ~/tools/samtools-1.17/samtools stats bowtie2_results/filtered_bam${file}_quality_controlled.bam > bowtie2_results/filtered_bam${file}_stats.txt


In [ ]:
# Run MultiQC
conda activate python3.7
multiqc ./bowtie2_results/filtered_bam/. -o ./bowtie2_results/filtered_bam/
conda deactivate

<img src="https://www.dropbox.com/scl/fi/zzloqmf1ndiuyuec8a43s/samtools_alignment_plot-7.png?rlkey=0pd4f4s6b4f0gv0738u03f5z4&st=uhr9zonq&raw=1" width="600" alt="samtools_filtering">

* High-quality BAM files were generated by applying stringent filters to remove mitochondrial reads, low-quality alignments, and technical duplicates.
* Robust number of uniquely mapped nuclear reads (typically $30\text{--}50\text{M}$ per sample) was retained. 
* While one sample shows a lower final read count, the overall cohort maintains sufficient depth and consistency for high-confidence peak calling and downstream regulatory analysis.

2b. Merge bam files

In [ ]:
#!/bin/bash

# Merge high-quality biological replicates into pooled BAM files. Re-index and generate statistics for condition-level libraries.

#SBATCH --cpus-per-task=8
#SBATCH --mem=8G
#SBATCH --output=logs/merge_%A_%a.out
#SBATCH --error=logs/merge_%A_%a.err

# Create the output directory
mkdir -p bowtie2_results/merged_bam

# Set the paths
samtool_path=~/tools/samtools-1.17/samtools
output_dir=bowtie2_results/merged_bam
input_dir=bowtie2_results/filtered_bam  

# Define the input BAM files for each group.
# The number of groups and the number of replicates per group can vary depending on the experiment.
# Add or remove groups and replicate BAM files as needed.

input_files[0]="$input_dir/Sample1_quality_controlled.bam $input_dir/Sample2_quality_controlled.bam $input_dir/Sample3_quality_controlled.bam $input_dir/Sample4_quality_controlled.bam"
input_files[1]="$input_dir/Sample5_quality_controlled.bam $input_dir/Sample6_quality_controlled.bam $input_dir/Sample7_quality_controlled.bam $input_dir/Sample8_quality_controlled.bam"
input_files[2]="$input_dir/Sample9_quality_controlled.bam $input_dir/Sample10_quality_controlled.bam $input_dir/Sample11_quality_controlled.bam $input_dir/Sample12_quality_controlled.bam"
input_files[3]="$input_dir/Sample13_quality_controlled.bam $input_dir/Sample14_quality_controlled.bam $input_dir/Sample15_quality_controlled.bam $input_dir/Sample16_quality_controlled.bam"
input_files[4]="$input_dir/Sample17_quality_controlled.bam $input_dir/Sample18_quality_controlled.bam $input_dir/Sample19_quality_controlled.bam $input_dir/Sample20_quality_controlled.bam"

# Define the output names for the merged BAM files.
# Ensure the number of entries here matches the number of groups defined above.

merged_names=(
    "Group1.bam"
    "Group2.bam"
    "Group3.bam"
    "Group4.bam"
    "Group5.bam"
)

# Get the array index to access the correct input files
ARRAY_INDEX=$((SLURM_ARRAY_TASK_ID - 1))

# Get the input files for the current task
CURRENT_input_files=(${input_files[${ARRAY_INDEX}]})

# Merge BAM files, index merged BAM file, generate statistics using samtools stats
MERGED_NAME="${merged_names[${ARRAY_INDEX}]}"
${samtool_path} merge ${output_dir}/${MERGED_NAME} ${CURRENT_input_files[@]}
${samtool_path} index ${output_dir}/${MERGED_NAME}
${samtool_path} stats ${output_dir}/${MERGED_NAME} > ${output_dir}/${MERGED_NAME}_stats.txt

In [ ]:
# Run MultiQC
conda activate python3.7
multiqc ./bowtie2_results/merged_bam/. -o ./bowtie2_results/merged_bam/
conda deactivate 

<img src="https://www.dropbox.com/scl/fi/xnoqd600lgoxmsas1u4ct/samtools_alignment_plot-8.png?rlkey=kkgxoox9q5ay7ekm30xkb8nqj&st=prjo2dr3&raw=1" width="600" alt="samtools_filtering">

* Each pooled condition achieved a substantial sequencing depth, ranging from approximately 140M to nearly 190M mapped reads.
* This high-depth aggregation provides the necessary statistical power to identify subtle chromatin accessibility changes across different treatment groups with high confidence.

#### Step 3: ATAC-specific QC (R Script)
Executes the R-based quality checks for fragment distribution.

In [ ]:
######## QC for alignments ----

library(ATACseqQC)
library(Rsamtools)
library(TxDb.Hsapiens.UCSC.hg38.knownGene)
library(BSgenome.Hsapiens.UCSC.hg38)
library(ggplot2)

## Define BAM directory
bam_dir <- "bowtie2_results/merged_bam"

## Identify BAM files and indexes
bamfile <- list.files(path = bam_dir,
                      pattern = "merged.bam$",
                      full.names = TRUE)

bamfile.index <- paste0(bamfile, ".bai")

## Extract sample labels automatically
bamfile.label <- gsub("_merged.bam", "", basename(bamfile))

In [ ]:
#################################################
## 1. Estimate library complexity
#################################################

lib_comp <- vector("list", length(bamfile))

for (i in seq_along(bamfile)) {

  dup_freq <- readsDupFreq(bamFile = bamfile[i],
                           index = bamfile.index[i])

  lib_comp[[i]] <- estimateLibComplexity(dup_freq)

}

names(lib_comp) <- bamfile.label
lib_comp[[1]]

<img src="https://www.dropbox.com/scl/fi/ecx77zrjvg5dn3uywv6xl/IL13_lib_comp.png?rlkey=gckloslymgfr6s8fkx4hkgbpo&st=a23kng8a&raw=1" width="600" alt="library_complexity">

* Library complexity analysis confirms high sequencing diversity, with 90.8M distinct fragments identified from 94.2M total reads. 
* The linear relationship in the estimation plot indicates the library is far from saturation, with a projected yield of 1.5B unique fragments at 20x depth.

In [ ]:
#################################################
## 2. Fragment size distribution
#################################################

frag_size <- vector("list", length(bamfile))

for (i in seq_along(bamfile)) {

  frag_size[[i]] <- fragSizeDist(
    bamFiles = bamfile[i],
    bamFiles.labels = bamfile.label[i],
    index = bamfile.index[i]
  )

}

names(frag_size) <- bamfile.label
frag_size[[1]]

<img src="https://www.dropbox.com/scl/fi/5m3fzr4lasktxiv9muokl/IL13_frag_size.png?rlkey=421a1133qh0bgakfgo5u6fcap&st=wfsx2i48&raw=1" width="600" alt="fragment_size">

* **Nucleosome-Free Regions (NFR)**: A prominent peak is observed at <100 bp, representing accessible DNA between nucleosomes.
* **Nucleosomal Periodicities**: Distinct subsequent peaks at approximately 200, 400, and 600 bp correspond to mono-, di-, and tri-nucleosome protected fragments, respectively.
* **Library Integrity**: The clear separation between these peaks confirms successful transposition and suggests the chromatin structure remained intact during preparation.


In [ ]:
#################################################
## 3. Mapping Quality Control
#################################################

bamfileQC <- vector("list", length(bamfile))

for (i in seq_along(bamfile)) {

  bamfileQC[[i]] <- bamQC(
    bamfile = bamfile[i],
    index = bamfile.index[i],
    mitochondria = "chrM",
    outPath = NULL,
    doubleCheckDup = TRUE
  )

}

names(bamfileQC) <- bamfile.label

## Coefficients
bamfileQC[[1]]$totalQNAMEs
[1] 94250179

bamfileQC[[1]]$duplicateRate
[1] 0.02001664

bamfileQC[[1]]$properPairRate
[1] 1

bamfileQC[[1]]$nonRedundantFraction
[1] 0.9577745

bamfileQC[[1]]$PCRbottleneckCoefficient_1
[1] 0.9794527

bamfileQC[[1]]$PCRbottleneckCoefficient_2
[1] 52.63026

* **totalQNAMEs**: Count of 94,250,179 indicates a very high-depth library, provides the exceptional resolution required for advanced analyses like transcription factor footprinting.
* **duplicateRate**: Extremely low duplication rate indicating good library complexity and minimal PCR amplification bias.
* **properPairRate**: 100% of reads align as proper pairs indicating excellent alignment quality.
* **nonRedundantFraction**: Unique reads/Total reads, a value of 0.95 indicates very high library complexity.
* **PCRbottleneckCoefficient_1**: Values less than 0.7 indicate severe bottlenecking, 0.7 and 0.9 indicate moderate bottlenecking, and Greater than 0.9 show no bottlenecking.
* **PCRbottleneckCoefficient_2**: Values less than 1 indicate severe bottlenecking, and between 1 and 3 indicate moderate bottlenecking. Greater than 3 show no bottlenecking.

In [ ]:
## MAPQ distribution plots
ggplot(bamfileQC[[1]]$MAPQ, aes(x = Var1, y = Freq, fill = Freq)) +
    geom_bar(stat = "identity") +
    labs(x = "MAPQ score", y = "Read count") +
    theme_minimal()

<img src="https://www.dropbox.com/scl/fi/t5f5vnxsscomiqnubufq3/IL13_MAPQ.png?rlkey=5fecsx7hj19c7p3xcv4wr0r62&st=xke8f1sj&raw=1" width="600" alt="fragment_size">

* **High-Confidence Mapping**: The vast majority of reads exhibit a MAPQ score of 42, indicating a very high probability that these reads are uniquely and correctly mapped to the reference genome.
* **Effective Filtering**: Since the plot starts at a MAPQ of 10, it confirms that low-quality, multi-mapping reads (which typically have MAPQ scores of 0–2) were successfully removed during the filtering stage.
* **Minimal Noise**: The presence of only negligible counts for scores between 10 and 40 suggests that the sequencing and alignment processes were highly specific, with very few ambiguous alignments retained.

#### Step 4: Signal Visualization and Profiling with deepTools

Generate normalized coverage tracks and summary plots across genomic regions such as peaks or transcription start sites (TSS).

In [ ]:
# Run deeptools
conda activate deeptools

**4a. bamCoverage**  

This tool takes an alignment of reads or fragments as input (BAM file) and generates a coverage track (bigWig or bedGraph) as output. The coverage is calculated as the number of reads per bin, where bins are short consecutive counting windows of a defined size.

In [ ]:
#!/bin/bash
# Convert bam file to a normalized bigwig using the bamCoverage tool in deepTools

#SBATCH --cpus-per-task=8
#SBATCH --mem=16G            
#SBATCH --time=12:00:00      
#SBATCH --output=./bam_coverage_%A_%a.out
#SBATCH --error=./bam_coverage_%A_%a.err

# Define an array of file names to be processed
readarray -t files < <(ls bowtie2_results/merged_bam/*_merged.bam | sed 's/.*bowtie2_results\/merged_bam\///' | sed 's/_merged.bam.*//' | sort -u)

# Get the file name for this array task
file="${files[$SLURM_ARRAY_TASK_ID - 1]}"

# Generate coverage track (bigWig or bedGraph)
srun bamCoverage --bam bowtie2_results/merged_bam/${file}_merged.bam --outFileName bowtie2_results/merged_bam/${file}.bw --binSize 1 --normalizeUsing RPKM --effectiveGenomeSize 2913022398 --numberOfProcessors 8 --verbose 

**4b: computeMatrix**

This tool calculates scores per genome regions and prepares an intermediate file that can be used with plotHeatmap and plotProfiles. Typically, the genome regions are genes, but any other regions defined in a BED file can be used. computeMatrix accepts multiple score files (bigWig format) and multiple regions files (BED format). 

In [ ]:
#!/bin/bash
# Generate a data matrix of read counts over your peak regions using the deepTools computeMatrix function

#SBATCH --cpus-per-task=8
#SBATCH --mem=16G            
#SBATCH --time=12:00:00      
#SBATCH --output=./compute_matrix_%A_%a.out
#SBATCH --error=./compute_matrix_%A_%a.err

# Define an array of file names to be processed
readarray -t files < <(ls bowtie2_results/merged_bam/*_merged.bam | sed 's/.*bowtie2_results\/merged_bam\///' | sed 's/_merged.bam.*//' | sort -u)

# Get the file name for this array task
file="${files[$SLURM_ARRAY_TASK_ID - 1]}"

# Calculate scores per genome regions and prepare an intermediate file that can be used with plotHeatmap and plotProfiles
srun computeMatrix reference-point -S bowtie2_results/merged_bam/${file}.bw -R macs2/${file}/${file}_summits.bed --outFileName bowtie2_results/merged_bam/${file}.matrix.gz --referencePoint TSS --beforeRegionStartLength 3000 --afterRegionStartLength 3000  --binSize 1 --numberOfProcessors max --verbose

**4c: plotHeatmap**

This tool creates a heatmap for scores associated with genomic regions. The program requires a matrix file generated by the tool computeMatrix.

In [ ]:
#!/bin/bash
# Generate a heatmap and average plot from the data matrix using the plotHeatmap function in deepTools

#SBATCH --cpus-per-task=8
#SBATCH --mem=16G            
#SBATCH --output=./plot_heatmap_%A_%a.out
#SBATCH --error=./plot_heatmap_%A_%a.err

# Define an array of file names to be processed
readarray -t files < <(ls bowtie2_results/merged_bam/*_merged.bam | sed 's/.*bowtie2_results\/merged_bam\///' | sed 's/_merged.bam.*//' | sort -u)

# Get the file name for this array task
file="${files[$SLURM_ARRAY_TASK_ID - 1]}"

# Create heatmap for scores associated with genomic regions
srun plotHeatmap --matrixFile bowtie2_results/merged_bam/${file}.matrix.gz --outFileName bowtie2_results/merged_bam/${file}_heatmap.png --colorMap=Blues --verbose

<img src="https://www.dropbox.com/scl/fi/bxe0vwggil7676dza88zx/IL13_heatmap.png?rlkey=cdw07b3gl6agar7kfhx25tzoq&st=0uzlb7r0&raw=1" width="175" alt="plotHeatmap">

* **Strong TSS Enrichment**: A sharp, symmetric peak at the TSS indicates high chromatin accessibility at active promoters.
* **High Signal-to-Noise**: The dark central band against a clean background validates successful mitochondrial and MAPQ filtering.
* **Robust Library Depth**: Intensity scores reaching 2000 RPKM reflect your high sequencing depth (~94M reads) and 0.95 non-redundant fraction.
* **Optimal Resolution**: The clear visualization confirms that --binSize 10 preserved biological signal while improving computational efficiency.

**4d: plotProfile**

This tool creates a profile plot for scores over sets of genomic regions. Typically, these regions are genes, but any other regions defined in BED will work. A matrix generated by computeMatrix is required.

In [ ]:
#!/bin/bash
# Creates a profile plot for scores over sets of genomic regions using the plotProfile function in deepTools

#SBATCH --cpus-per-task=8
#SBATCH --mem=16G            
#SBATCH --output=./plot_profile_%A_%a.out
#SBATCH --error=./plot_profile_%A_%a.err

# Define an array of file names to be processed
readarray -t files < <(ls bowtie2_results/merged_bam/*_merged.bam | sed 's/.*bowtie2_results\/merged_bam\///' | sed 's/_merged.bam.*//' | sort -u)

# Get the file name for this array task
file="${files[$SLURM_ARRAY_TASK_ID - 1]}"

# Create profile plot for scores over sets of genomic regions
srun plotProfile --matrixFile bowtie2_results/merged_bam/${file}.matrix.gz --outFileName bowtie2_results/merged_bam/${file}_profile.png --verbose

<img src="https://www.dropbox.com/scl/fi/rqp2axsag7los29qhdki8/IL13_profile.png?rlkey=qxedh9qq7pkio3stizhhhlwwo&st=krjtog8b&raw=1" width="400" alt="plotProfile">

* **Precise TSS Enrichment**: The plot demonstrates a sharp, high-intensity peak centered exactly at the Transcription Start Site (TSS), which is a hallmark of high-quality ATAC-seq data.
* **Signal Magnitude**: The peak reaches approximately 2000 RPKM, indicating robust sequencing depth and clear detection of open chromatin regions within the IL13 group.
* **Low Flanking Noise**: The baseline signal remains consistently low (near zero) at the $\pm 3$ kb boundaries, confirming that the Step 2 filtering of * mitochondrial and low-MAPQ reads was highly effective.

#### Summary of Observations
* **Data Integrity**: High library quality is confirmed by a dominant MAPQ 42 and a 0.02 duplicate rate. Rigorous filtering of mitochondrial and low-quality reads (MAPQ < 10) has successfully produced a high-complexity, non-redundant nuclear DNA signal (PBC1 = 0.979).
* **Computational Efficiency**: Utilizing array tasks reduced total wall-clock time for multi-sample batch.
* **Next Steps**: Identifying peaks and validate reliability by quantifying FRiP while excluding ENCODE blacklist regions. Mapping high confidence peaks to promoters, introns, and distal intergenic regions to characterize the regulatory landscape.